In [ ]:
!pip install -r ..\..\requirements.txt

In [ ]:
1

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from drawdata import ScatterWidget
from sqlalchemy import create_engine
import dotenv
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
import iris
import json

dotenv.load_dotenv()

connection_params = {
    "hostname": os.getenv("IRIS_SERVER"),
    "port": int(os.getenv("IRIS_PORT")),
    "namespace": os.getenv("IRIS_NAMESPACE"),
    "username": os.getenv("IRIS_USERNAME"),
    "password": os.getenv("IRIS_PASSWORD")
}


### Create Points

In [ ]:
datetime_now = datetime.now(tz=ZoneInfo(os.getenv("TZ")))
datetime_now_str = datetime_now.strftime("%Y-%m-%d %H:%M:%S")
widget = ScatterWidget(height=200, width=200)
widget

In [ ]:
df = widget.data_as_pandas[["x", "y", "label"]]
df.head()

In [ ]:
engine = create_engine(f"iris://{connection_params['username']}:{connection_params['password']}@{connection_params['hostname']}:{connection_params['port']}/{connection_params['namespace']}")
df.to_sql("PointSamples", engine, schema="MLpipeline", if_exists='append', index=False, method='multi')

### Prediction Testing

In [ ]:
date_filter = f"datetime >= '{datetime_now_str}'"
with iris.connect(**connection_params) as conn:
    iris_obj = iris.createIRIS(conn)
    json_result  = iris_obj.classMethodValue("MLpipeline.PredictionService", "Predict", date_filter)
    predictions = json.loads(json_result)

# PLOT OF RESULTS JUST GOT FROM IRIS
mae = mean_absolute_error(df.y, predictions)
r2 = r2_score(df.y, predictions)
print(f"Mean Absolute Error: {mae:.4f}")
print(f"R^2 Score: {r2:.4f}")

plt.Figure(figsize=(10,10))
plt.scatter(df.x, df.y, cmap='viridis', label='Data Points')
plt.scatter(df.x, predictions, marker='x', label='Predictions')
plt.xlim(0, 200)
plt.ylim(0, 200)
plt.xlabel('x')
plt.ylabel('y / y_pred')
plt.title(f'Data Points and Model Predictions. MAE: {mae:.3f}, R²: {r2:.3f}')
plt.legend()
plt.grid()
plt.show()

### Monitoring Testing

In [ ]:
with iris.connect(**connection_params) as conn:
    iris_obj = iris.createIRIS(conn)
    PM = iris_obj.classMethodObject("MLpipeline.PerformanceMonitoring", "%New")
    status = PM.invoke("MetricsMonitoring",datetime_now_str)
print(f"Performance monitoring status: {status}")